In [1]:
"""
Iterative VIF Feature Elimination
==================================
Removes the highest-VIF feature one at a time until all remaining
features have VIF <= threshold (default: 10).

"""
 
import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.svm import SVC
from interpret.glassbox import ExplainableBoostingClassifier
from sklearn.metrics import (
    confusion_matrix, f1_score, precision_score, recall_score,
    accuracy_score, roc_auc_score, roc_curve
)
from sklearn.base import clone
from sklearn.utils.class_weight import compute_sample_weight

In [2]:
DATA_PATH      = "Data for Task 1.csv"   
TARGET_COL     = "diagnosis"              
TARGET_POS     = "M"                    
DROP_COLS      = ["id", "Unnamed: 32"]  
VIF_THRESHOLD  = 10                      
CV_FOLDS       = 5                       
RANDOM_STATE   = 42

In [3]:
def compute_vif(df: pd.DataFrame) -> dict[str, float]:
    """Return a {feature: VIF} dict for all columns in df."""
    arr = df.values.astype(float)
    return {
        col: variance_inflation_factor(arr, i)
        for i, col in enumerate(df.columns)
    }
 
 
def compute_auc(X: pd.DataFrame, y: pd.Series) -> float:
    """5-fold cross-validated AUC of a standard logistic regression."""
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    lr = LogisticRegression(max_iter=1_000, random_state=RANDOM_STATE)
    scores = cross_val_score(lr, Xs, y, cv=CV_FOLDS, scoring="roc_auc")
    return float(scores.mean())

In [4]:

def run_elimination(
    data_path: str = DATA_PATH,
    target_col: str = TARGET_COL,
    target_pos: str = TARGET_POS,
    drop_cols: list[str] = DROP_COLS,
    vif_threshold: float = VIF_THRESHOLD,
) -> tuple[list[str], pd.DataFrame]:

    # Load & prepare
    df = pd.read_csv(data_path)
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])
 
    y = (df[target_col] == target_pos).astype(int)
    X = df.drop(columns=[target_col])
 
    log_rows = []
    step = 0
 
    print(f"Starting elimination | {len(X.columns)} features | VIF threshold = {vif_threshold}\n")
    print(f"{'Step':<5} {'Features':>8} {'Dropped':<30} {'Max VIF':>12} {'AUC':>8}")
    print("-" * 70)
 
    while True:
        vif       = compute_vif(X)
        max_feat  = max(vif, key=vif.get)
        max_val   = vif[max_feat]
        auc       = compute_auc(X, y)
 
        log_rows.append({
            "step":            step,
            "n_features":      len(X.columns),
            "feature_dropped": max_feat if max_val > vif_threshold else None,
            "max_vif_before":  round(max_val, 2),
            "auc":             round(auc, 4),
        })
 
        drop_label = max_feat if max_val > vif_threshold else "— stop —"
        print(f"{step:<5} {len(X.columns):>8} {drop_label:<30} {max_val:>12,.1f} {auc:>8.4f}")
 
        if max_val <= vif_threshold:
            break
 
        X    = X.drop(columns=[max_feat])
        step += 1
 
    log = pd.DataFrame(log_rows)
    final_features = list(X.columns)
 
    print("\n" + "=" * 70)
    print(f"Elimination complete — {len(final_features)} features remaining:\n")
    for feat in final_features:
        vif_val = compute_vif(X)[feat]
        print(f"  {feat:<35} VIF = {vif_val:.2f}")
 
    print(f"\nAUC with all features : {log.iloc[0]['auc']:.4f}")
    print(f"AUC with {len(final_features)} features   : {log.iloc[-1]['auc']:.4f}")
    print(f"AUC loss              : {log.iloc[0]['auc'] - log.iloc[-1]['auc']:.4f}")
 
    return final_features, log
 

In [5]:
if __name__ == "__main__":
    final_features, log = run_elimination()
 
    # Save the elimination log
    log.to_csv("vif_elimination_log.csv", index=False)
    print("\nElimination log saved to: vif_elimination_log.csv")
 
    # Optionally: save the reduced dataset
    df = pd.read_csv(DATA_PATH)
    df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])
    reduced = df[[TARGET_COL] + final_features]
    reduced.to_csv("VIF_data_reduced.csv", index=False)
    print("Reduced dataset saved to: VIF_data_reduced.csv")

Starting elimination | 30 features | VIF threshold = 10

Step  Features Dropped                             Max VIF      AUC
----------------------------------------------------------------------
0           30 radius_mean                        63,306.2   0.9953
1           29 radius_worst                        7,573.9   0.9953
2           28 perimeter_mean                      3,901.9   0.9951
3           27 perimeter_worst                       668.4   0.9951
4           26 fractal_dimension_mean                508.1   0.9943
5           25 smoothness_worst                      368.1   0.9946
6           24 texture_worst                         309.5   0.9952
7           23 fractal_dimension_worst               184.7   0.9942
8           22 symmetry_worst                        167.3   0.9942
9           21 concavity_mean                        142.3   0.9923
10          20 radius_se                             105.0   0.9927
11          19 concave points_worst                  100

In [6]:
df_reduced = pd.read_csv("VIF_data_reduced.csv")
df_reduced = df_reduced.loc[:, ~df_reduced.columns.str.contains("^Unnamed")]

y = df_reduced["diagnosis"]
X = df_reduced.drop(columns=["diagnosis"])


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

lr = LogisticRegression(
    max_iter=10000,
    random_state=42,
    class_weight="balanced"
)

lr.fit(X_train_sc, y_train)

y_pred = lr.predict(X_test_sc)
cm = confusion_matrix(y_test, y_pred)

cv_f1 = cross_val_score(
    lr,
    X_train_sc,
    y_train,
    cv=cv,
    scoring="f1_macro"
).mean()

In [8]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

def evaluate_model(name, model, Xtr, Xte, use_sample_weight=False):

    if use_sample_weight:
        sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)
        model.fit(Xtr, y_train, sample_weight=sample_weight)
    else:
        model.fit(Xtr, y_train)

    y_pred = model.predict(Xte)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(Xte)[:, 1]
    else:
        y_prob = model.decision_function(Xte)

    cm = confusion_matrix(y_test, y_pred, labels=["B", "M"])

    cv_scores = []

    for train_idx, val_idx in cv.split(Xtr, y_train):
        model_cv = clone(model)

        if isinstance(Xtr, np.ndarray):
            X_fold_train = Xtr[train_idx]
            X_fold_val = Xtr[val_idx]
        else:
            X_fold_train = Xtr.iloc[train_idx]
            X_fold_val = Xtr.iloc[val_idx]

        y_fold_train = y_train.iloc[train_idx]
        y_fold_val = y_train.iloc[val_idx]

        if use_sample_weight:
            fold_weight = compute_sample_weight(
                class_weight="balanced",
                y=y_fold_train
            )
            model_cv.fit(X_fold_train, y_fold_train, sample_weight=fold_weight)
        else:
            model_cv.fit(X_fold_train, y_fold_train)

        y_fold_pred = model_cv.predict(X_fold_val)
        cv_scores.append(f1_score(y_fold_val, y_fold_pred, pos_label="M"))

    cv_scores = np.array(cv_scores)

    return {
        "name": name,
        "model": model,
        "f1": f1_score(y_test, y_pred, pos_label="M"),
        "recall": recall_score(y_test, y_pred, pos_label="M"),
        "precision": precision_score(y_test, y_pred, pos_label="M"),
        "accuracy": accuracy_score(y_test, y_pred),
        "auc": roc_auc_score((y_test == "M").astype(int), y_prob),
        "cv_f1": cv_scores.mean(),
        "cv_std": cv_scores.std(),
        "fn": cm[1, 0],
        "fp": cm[0, 1],
        "cm": cm,
        "y_pred": y_pred,
        "y_prob": y_prob
    }

models_to_run = {
    "Logistic Regression (balanced, 21 features)": (
        LogisticRegression(
            max_iter=10000,
            class_weight="balanced",
            random_state=42
        ),
        X_train_sc,
        X_test_sc,
        False
    ),

    "Shallow Decision Tree (balanced, 21 features)": (
        DecisionTreeClassifier(
            max_depth=5,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42
        ),
        X_train_sc,
        X_test_sc,
        False
    ),

"Explainable Boosting Machine (balanced, 21 features)": (
    ExplainableBoostingClassifier(
        interactions=0,
        outer_bags=8,
        max_rounds=300,
        random_state=42
    ),
    X_train,
    X_test,
    True
),

    "Linear SVM (balanced, 21 features)": (
        SVC(
            kernel="linear",
            probability=True,
            class_weight="balanced",
            random_state=42
        ),
        X_train_sc,
        X_test_sc,
        False
    )
}


results_21_balanced = {}

print(f"\n  {'Model':<55} {'F1':>7} {'Recall':>7} {'Prec':>7} {'AUC':>7} {'CV F1':>8} {'FN':>4} {'FP':>4}")
print("  " + "-" * 110)

for name, (model, Xtr, Xte, use_sample_weight) in models_to_run.items():
    result = evaluate_model(name, model, Xtr, Xte, use_sample_weight)
    results_21_balanced[name] = result

    flag = " ✓" if result["f1"] > 0.95 else ""

    print(
        f"  {name:<55} "
        f"{result['f1']:>7.4f} "
        f"{result['recall']:>7.4f} "
        f"{result['precision']:>7.4f} "
        f"{result['auc']:>7.4f} "
        f"{result['cv_f1']:>8.4f} "
        f"{result['fn']:>4} "
        f"{result['fp']:>4}"
        f"{flag}"
    )


  Model                                                        F1  Recall    Prec     AUC    CV F1   FN   FP
  --------------------------------------------------------------------------------------------------------------
  Logistic Regression (balanced, 21 features)              0.9412  0.9524  0.9302  0.9950   0.9212    2    3
  Shallow Decision Tree (balanced, 21 features)            0.9157  0.9048  0.9268  0.9529   0.8729    4    3
  Explainable Boosting Machine (balanced, 21 features)     0.9512  0.9286  0.9750  0.9921   0.9026    3    1 ✓
  Linear SVM (balanced, 21 features)                       0.9412  0.9524  0.9302  0.9937   0.9252    2    3
